In [2]:
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
import requests

In [2]:
base_url = "https://firestore.googleapis.com/v1/projects/oeskb-app/databases/(default)/documents/saisonen/"

seasons = ["2021_2022", "2022_2023", "2023_2024", "2024_2025", "2025_2026"]

leagues  = {
    "SL Damen": "1_8_",
    "BL Damen": "1_7_",
    # "SL Herren": "1_6_",
    # "BL Herren Ost": "1_5_",
    # "BL Herren West": "1_4_",
}

def get_games_data(games_url):
    response = requests.get(games_url)
    data = response.json()

    games = data['fields']['spiele']['mapValue']['fields']
    game_keys = list(games.keys())

    game_data = pd.DataFrame()

    for game_key in game_keys:
        game = games[game_key]["mapValue"]["fields"]["infos"]["mapValue"]["fields"]
        game_series = pd.json_normalize(game)
        game_series['spiel_id'] = game_key
        game_data = pd.concat([game_data, game_series], ignore_index=True)

    game_data = game_data.rename(columns={
        'datum.stringValue': 'datum',
        'uhrzeit.stringValue': 'uhrzeit',
        'bahn.stringValue': 'bahn',
    })

    game_data = game_data[["datum", "uhrzeit", "bahn", "spiel_id"]]

    return game_data

def get_results_and_player_data(results_url):
    response = requests.get(results_url)
    data_results = response.json()

    results = data_results['fields']['spiele']['mapValue']['fields']
    game_keys = list(results.keys())

    result_data = pd.DataFrame()
    for game in game_keys:
        for location in ['heim', 'gast']:

            players = results[game]['mapValue']['fields'][location]['mapValue']['fields']['spieler']['mapValue']['fields']
            player_keys = list(players.keys())

            for player_key in player_keys:
                player = players[player_key]['mapValue']['fields']
                player_series = pd.json_normalize(player)
                player_series['spieler_id'] = player_key
                player_series['heim'] = location == 'heim'
                player_series['spiel_id'] = game
                result_data = pd.concat([result_data, player_series], ignore_index=True)

    # if integer value is not set, take double value
    result_data['sp'] = result_data['sp.integerValue'].fillna(result_data['sp.doubleValue'])

    if 'mp.doubleValue' in result_data.columns:
        result_data['mp'] = result_data['mp.doubleValue'].fillna(result_data['mp.integerValue'])
    else:
        result_data['mp'] = result_data['mp.integerValue']

    results = result_data[["spieler_id", "spiel_id", "kegel.integerValue", "sp", "mp", "heim", "bahn.integerValue", "dg.integerValue"]]
    results = results.rename(columns={
        'kegel.integerValue': 'kegel',
        'bahn.integerValue': 'startbahn',
        'dg.integerValue': 'dg',
    })


    player_data = result_data.drop_duplicates(subset=['spieler_id'])

    player_data = player_data.rename(columns={
        'land.stringValue': 'land',
        'name.stringValue': 'name',
        'akJahr.integerValue': 'jahrgang',
    })

    player_data = player_data[["spieler_id", "name", "jahrgang", "land"]]
    player_data[['passnummer', 'name']] = player_data['name'].str.split(' / ', n=1, expand=True)

    return results, player_data


all_games = pd.DataFrame()
all_players = pd.DataFrame()

for season in seasons:
    for league, id in leagues.items():
        games_url = base_url + id + season
        results_url_H = games_url + "/schnittliste/" + id + season + "H"
        results_url_F = games_url + "/schnittliste/" + id + season + "F"

        print(f"Processing {season} - {league}")

        # Get game data
        game_data = get_games_data(games_url)
        game_data['liga'] = league
        
        # Get results data
        results_H , player_H = get_results_and_player_data(results_url_H)

        if season != "2025_2026":
            results_F, player_F = get_results_and_player_data(results_url_F)

        results = pd.concat([results_H, results_F], ignore_index=True)
        player = pd.concat([player_H, player_F], ignore_index=True)
        player = player.drop_duplicates(subset=['spieler_id'])

        if "Damen" in league:
            player['geschlecht'] = 'w'
        else:
            player['geschlecht'] = 'm'

        # concat game data with results
        game_data = pd.merge(results, game_data , on='spiel_id', how='left')

        all_games = pd.concat([all_games, game_data], ignore_index=True)
        all_players = pd.concat([all_players, player], ignore_index=True)

# save to csv
all_games.to_csv("all_games.csv", index=False)
all_players.to_csv("all_players.csv", index=False)

Processing 2021_2022 - SL Damen
Processing 2021_2022 - BL Damen
Processing 2021_2022 - SL Herren
Processing 2021_2022 - BL Herren Ost
Processing 2021_2022 - BL Herren West
Processing 2022_2023 - SL Damen
Processing 2022_2023 - BL Damen
Processing 2022_2023 - SL Herren
Processing 2022_2023 - BL Herren Ost
Processing 2022_2023 - BL Herren West
Processing 2023_2024 - SL Damen
Processing 2023_2024 - BL Damen
Processing 2023_2024 - SL Herren
Processing 2023_2024 - BL Herren Ost
Processing 2023_2024 - BL Herren West
Processing 2024_2025 - SL Damen
Processing 2024_2025 - BL Damen
Processing 2024_2025 - SL Herren
Processing 2024_2025 - BL Herren Ost
Processing 2024_2025 - BL Herren West
Processing 2025_2026 - SL Damen
Processing 2025_2026 - BL Damen
Processing 2025_2026 - SL Herren
Processing 2025_2026 - BL Herren Ost
Processing 2025_2026 - BL Herren West


In [3]:
all_games = pd.read_csv('games.csv')
all_players = pd.read_csv('players.csv')

player_games = pd.merge(all_games, all_players, on='spieler_id', how='left')

player_games.to_csv("player_games_joined.csv", index=False)

In [4]:
# convert kegel, mp to numeric
all_games['kegel'] = pd.to_numeric(all_games['kegel'])
all_games['mp'] = pd.to_numeric(all_games['mp'])
all_games['sp'] = pd.to_numeric(all_games['sp'])
all_games['datum'] = pd.to_datetime(all_games['datum'], format='%Y-%m-%d')

player_games = pd.merge(all_games, all_players, on='spieler_id', how='left')
player_games = player_games.sort_values(by=['spieler_id', 'datum'])


In [5]:
# show specific game
player_games[player_games['spiel_id'] == 'bvmbagb4MMVzDB9swvUs'].drop_duplicates("spieler_id")

,spieler_id,spiel_id,kegel,sp,mp,heim,startbahn,dg,datum,uhrzeit,bahn,liga,name,jahrgang,land,passnummer,geschlecht
4607,0qOdBd9JktVeRbh6wNfA,bvmbagb4MMVzDB9swvUs,475,0.5,0.0,True,1.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,HAUSER Judith,1953.0,AUT,116042,w
4629,5DQqneYxSJ56TQAiLtbi,bvmbagb4MMVzDB9swvUs,487,3.0,1.0,False,2.0,2,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,KLEBER Elisabeth,1985.0,AUT,186027,w
4619,63tw4ipYfAkdLy9yqxGR,bvmbagb4MMVzDB9swvUs,472,2.0,0.0,False,4.0,2,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,TORREMANTE Caroline,1978.0,GER,186050,w
4605,CEXIqZvzmIgYJc0Vlucd,bvmbagb4MMVzDB9swvUs,476,2.0,1.0,True,3.0,2,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,PFEIFER Julia,1986.0,AUT,116071,w
4624,HFdZhWlRyDDgubstJkvF,bvmbagb4MMVzDB9swvUs,500,4.0,1.0,False,2.0,1,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,NIKOLIC Maja Katharina,2002.0,AUT,186012,w
4638,MvwoeQ09FFZZwdrPk7l4,bvmbagb4MMVzDB9swvUs,534,3.5,1.0,False,2.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
4612,QUUikB9JxnWtuLXNEODJ,bvmbagb4MMVzDB9swvUs,434,1.0,0.0,True,1.0,2,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,BLANK Sabine,1965.0,AUT,116016,w
4616,UwOYtuLTQMVrrb4JWMtn,bvmbagb4MMVzDB9swvUs,475,4.0,1.0,False,4.0,1,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,ZUMTOBEL Ursula,1968.0,AUT,186111,w
4597,YIv16tENhq1u5luHDzTl,bvmbagb4MMVzDB9swvUs,445,0.0,0.0,True,1.0,1,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,RENNER Nada,1957.0,AUT,116013,w
4602,Ybh8QOnsSSuixnqqJqUT,bvmbagb4MMVzDB9swvUs,499,2.5,1.0,True,3.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,EDER Marianne,1955.0,AUT,116026,w


In [6]:
# player_games.sort_values(by=['datum'], inplace=True, ascending=False)
player_games[player_games['spieler_id'] == 'MvwoeQ09FFZZwdrPk7l4']

,spieler_id,spiel_id,kegel,sp,mp,heim,startbahn,dg,datum,uhrzeit,bahn,liga,name,jahrgang,land,passnummer,geschlecht
4638,MvwoeQ09FFZZwdrPk7l4,bvmbagb4MMVzDB9swvUs,534,3.5,1.0,False,2.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
4639,MvwoeQ09FFZZwdrPk7l4,bvmbagb4MMVzDB9swvUs,534,3.5,1.0,False,2.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
4640,MvwoeQ09FFZZwdrPk7l4,bvmbagb4MMVzDB9swvUs,534,3.5,1.0,False,2.0,3,2021-09-18,14:00,LSZ VIVA Steinbrunn,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
3906,MvwoeQ09FFZZwdrPk7l4,5D97YQ6Myu1aGZjNDzG1,567,3.0,1.0,True,3.0,3,2021-10-02,15:30,Güterbahnhof Wolfurt,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
3907,MvwoeQ09FFZZwdrPk7l4,5D97YQ6Myu1aGZjNDzG1,567,3.0,1.0,True,3.0,3,2021-10-02,15:30,Güterbahnhof Wolfurt,BL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60178,MvwoeQ09FFZZwdrPk7l4,cWcBRbIvK01te5CUeyO8,574,4.0,1.0,False,4.0,3,2024-03-09,13:00,Blaue Lagune,SL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
60179,MvwoeQ09FFZZwdrPk7l4,cWcBRbIvK01te5CUeyO8,574,4.0,1.0,False,4.0,3,2024-03-09,13:00,Blaue Lagune,SL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
61760,MvwoeQ09FFZZwdrPk7l4,Udh5HKMhsObiCgNBOWNA,598,2.0,1.0,False,4.0,1,2024-03-23,11:00,ASKÖ SKZ,SL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w
61761,MvwoeQ09FFZZwdrPk7l4,Udh5HKMhsObiCgNBOWNA,598,2.0,1.0,False,4.0,1,2024-03-23,11:00,ASKÖ SKZ,SL Damen,ANNASENSL Jasmin,1993.0,GER,186024,w


In [5]:
# convert from long to wide format
player_games_wide = player_games.pivot_table(
    index='spieler_id',
    columns='datum',
    values='kegel',
    aggfunc='first'
).reset_index()

player_games_wide

# player_scores_dict = player_games_wide.set_index('spieler_id').T.to_dict(orient='list')
# player_scores_dict = {k: [x for x in v if pd.notna(x)] for k, v in player_scores_dict.items()}
# player_scores_dict

datum,spieler_id,2021-09-11 00:00:00,2021-09-12 00:00:00,2021-09-18 00:00:00,2021-09-25 00:00:00,2021-10-02 00:00:00,2021-10-09 00:00:00,2021-10-16 00:00:00,2021-10-17 00:00:00,2021-10-23 00:00:00,...,2025-10-04 00:00:00,2025-10-11 00:00:00,2025-10-15 00:00:00,2025-10-18 00:00:00,2025-10-25 00:00:00,2025-10-31 00:00:00,2025-11-02 00:00:00,2025-11-08 00:00:00,2025-11-15 00:00:00,2025-11-22 00:00:00
0,02FJ4iQD3ERrO82AIvCX,NaN,NaN,NaN,548.0,485.0,NaN,NaN,NaN,530.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,06IEI2JFGzapPB7TWnBN,540.0,NaN,481.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0CoTEvszggXwcbjItnWm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0OZYwSXDry5xx4A18YL4,434.0,NaN,NaN,NaN,453.0,NaN,422.0,NaN,471.0,...,393.0,501.0,NaN,NaN,476.0,NaN,NaN,458.0,NaN,NaN
4,0Q82PcVArf0f3nMr61gB,539.0,NaN,550.0,506.0,NaN,NaN,528.0,NaN,429.0,...,521.0,528.0,NaN,NaN,473.0,NaN,NaN,550.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745,zmcYKQ1IjbRiXbLqPZJY,507.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
746,znuzhDJImx0v1Xou2geF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
747,zplBFZlBUgDSxa66o6Lh,568.0,NaN,558.0,618.0,564.0,565.0,NaN,NaN,NaN,...,596.0,558.0,NaN,NaN,587.0,NaN,NaN,540.0,630.0,NaN
748,zrTpHhDw3BUPlauI5560,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,517.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def random_aufstellung(heim, gast):
    heim_indices = np.random.permutation(len(heim))
    gast_indices = np.random.permutation(len(gast))

    return heim_indices, gast_indices


def evaluate_aufstellung(heim, gast, heim_indices, gast_indices, spiel_id):
    mp = 0

    for i in range(len(heim)):
        heim_player = heim.iloc[heim_indices[i]]
        heim_kegel = all_games[(all_games['spiel_id'] == spiel_id) & (all_games['spieler_id'] == heim_player)]['kegel'].values[0]

        gast_player = gast.iloc[gast_indices[i]]
        gast_kegel = all_games[(all_games['spiel_id'] == spiel_id) & (all_games['spieler_id'] == gast_player)]['kegel'].values[0]

        if gast_kegel > heim_kegel:
            mp += 1
    return mp


In [7]:
def win_probability(gast_scores, heim_scores):
    
    if not heim_scores or not gast_scores:
        return None
    
    heim_samples = np.random.choice(heim_scores, size=1000000, replace=True)
    gast_samples = np.random.choice(gast_scores, size=1000000, replace=True)

    return np.mean(gast_samples > heim_samples)

def win_prob_matrix(heim_scores, gast_scores):
    matrix = pd.DataFrame(index=gast_scores.keys(), columns=heim_scores.keys(), dtype=float)

    for gast, gast_score in gast_scores.items():
        for heim, heim_score in heim_scores.items():
            matrix.loc[gast, heim] = win_probability(gast_score, heim_score)

    return matrix

def prob_aufstellung(heim, gast):
    
    heim = heim.values
    gast = gast.values

    heim_scores = player_games_wide[player_games_wide['spieler_id'].isin(heim)].set_index('spieler_id').T.to_dict(orient='list')
    heim_scores = {k: [x for x in v if pd.notna(x)] for k, v in heim_scores.items()}

    gast_scores = player_games_wide[player_games_wide['spieler_id'].isin(gast)].set_index('spieler_id').T.to_dict(orient='list')
    gast_scores = {k: [x for x in v if pd.notna(x)] for k, v in gast_scores.items()}

    win_matrix = win_prob_matrix(heim_scores, gast_scores)
    row_ind, col_ind = linear_sum_assignment(-win_matrix)

    return row_ind, col_ind, win_matrix


def last_x_games_aufstellung(heim, gast, spiel_date, x=5):
    heim = heim.values
    gast = gast.values

    # get kegel scores of last x games before spiel_date
    heim_scores = {}
    for player in heim:
        scores = all_games[(all_games['spieler_id'] == player) & (all_games['datum'] < spiel_date)].sort_values(by='datum', ascending=False)['kegel'].values
        
        heim_scores[player] = scores[:x].tolist()

    # print(heim_scores)

    gast_scores = {}
    for player in gast:
        scores = all_games[(all_games['spieler_id'] == player) & (all_games['datum'] < spiel_date)].sort_values(by='datum', ascending=False)['kegel'].values
        gast_scores[player] = scores[:x].tolist()
    
    # print(gast_scores)

    win_matrix = win_prob_matrix(heim_scores, gast_scores)
    # print(win_matrix)
    row_ind, col_ind = linear_sum_assignment(-win_matrix)

    return row_ind, col_ind, win_matrix


# heim_indices, gast_indices = prob_aufstellung(heim, gast)
# for i in range(len(heim)):

#     print(heim_indices[i], gast_indices[i])

#     heim_player = heim.iloc(heim_indices[i])
#     gast_player = gast.iloc(gast_indices[i])
#     print(f"{heim_player} vs {gast_player}")

In [10]:
random_mp = 0
original_mp = 0
prob_mp = 0
last_mp = 0

num_games = 0

for x in [3, 5, 10, 20, 30]:
    points = 0
    games = 0

    for i, game in enumerate(all_games['spiel_id'].unique()):

        players = all_games[all_games['spiel_id'] == game]

        heim = players[players['heim'] == True]['spieler_id']
        gast = players[players['heim'] == False]['spieler_id']

        maja_id = 'HFdZhWlRyDDgubstJkvF'

        # if maja_id in gast.values:

        spiel_date = all_games[all_games['spiel_id'] == game]['datum'].values[0]
            
        # check for each player if they have at least x previous games
        if all(all_games[(all_games['spieler_id'] == player) & (all_games['datum'] < spiel_date)].shape[0] >= x for player in heim) and all(all_games[(all_games['spieler_id'] == player) & (all_games['datum'] < spiel_date)].shape[0] >= x for player in gast):

                # print(game)
                
            heim_indices, gast_indices, _ = last_x_games_aufstellung(heim, gast, spiel_date, x=x)
            points += evaluate_aufstellung(heim, gast, heim_indices, gast_indices, game)
            games += 1
                
    print(f"Last {x} Games: {points} points in {games} games")

            # if all players have at least x previous games

            # num_games += 1

            # heim_indices, gast_indices = random_aufstellung(heim, gast)
            # random_mp += evaluate_aufstellung(heim, gast, heim_indices, gast_indices, game)

            # original_mp += all_games[(all_games['spiel_id'] == game) & (all_games['heim'] == False)]['mp'].sum()

            # heim_indices, gast_indices,_ = prob_aufstellung(heim, gast)
            # prob_mp += evaluate_aufstellung(heim, gast, heim_indices, gast_indices, game)


            

# print(f"Average Match Points Random: {random_mp / num_games:.2f}")
# print(f"Average Match Points Original: {original_mp / num_games:.2f}")
# print(f"Average Match Points Probabilistic: {prob_mp / num_games:.2f}")
# for key, value in points.items():
#     print(f"Match Points Last {key} Games: {value}")

KeyboardInterrupt: 

In [8]:
gast = ["NIKOLIC Maja Katharina", "PRÖLL Tanja", "ZATSCHKOWITSCH Katharina", "NEMETH Enikö", "OBERHOFER Nina", "STÖCKL Sandra"]
# heim = ["BABICKA Renata", "HUBER Julia", "HAUER Dominique", "PETRU Thea", "EBNER Jessica", "MEISTERL Jasmin"]
heim = ["KUNOVA Nikola", "GSTREIN Franziska", "KOLLENDA Laura", "HAUSEGGER Andrea", "RABANSER Vera", "POCKSTALLER Katharina"]
# heim = ["HUDI Zsofia", "KUTI DR.JUR. Alexandra", "BLÜMEL Angelika", "STREBINGER Angela", "BROEKMANN Sabine", "WEISS Romana"]
# heim = ["WIEDERMANNOVA Hana", "MARTINKEVIC Dana", "RATHMAYER Dominique", "WALCH Sarah", "PLAMENIG Daniela", "PLAMENIG Nicole"]

# gast = ["JANDA Katrin", "SEBESTOVA Vendula", "KIRCHWEGER Gabriele", "LANGBAUER Christine", "WENCZLER Monika", "KUN Anna Eva"]

# gast = ["PAPP Laszlo", "KÖLLER Manuel", "SCHAUTZ Rudolf", "FOCHLER Stefan", "ROTH Michael", "SEIBERL Benjamin"]
# heim = ["NEMETH Laszlo", "SCHADELBAUER Florian", "SZÁSZ László Istvan", "FUCHS Dietmar", "BAUMGARTNER Thomas", "KRECHT Manfred"]

heim_ids = (
    player_games.loc[player_games['name'].isin(heim), ['name', 'spieler_id']]
    .drop_duplicates()
    .set_index('name')['spieler_id']
    .reindex(heim)   # keeps same order as your heim list
)

gast_ids = (
    player_games.loc[player_games['name'].isin(gast), ['name', 'spieler_id']]
    .drop_duplicates()
    .set_index('name')['spieler_id']
    .reindex(gast)  # keeps same order as your gast list
)

heim_map = (
    player_games.loc[player_games['name'].isin(heim), ['spieler_id', 'name']]
    .drop_duplicates()
    .set_index('spieler_id')['name']
    .to_dict()
)

gast_map = (
    player_games.loc[player_games['name'].isin(gast), ['spieler_id', 'name']]
    .drop_duplicates()
    .set_index('spieler_id')['name']
    .to_dict()
)

gast_indices, heim_indices, win_matrix = prob_aufstellung(heim_ids, gast_ids)
# gast_indices, heim_indices, win_matrix = last_x_games_aufstellung(heim_ids, gast_ids, "jlL73yf9DfNPKNtyLhBl")

# substituting ids with names
win_matrix.index = win_matrix.index.map(gast_map)
win_matrix.columns = win_matrix.columns.map(heim_map)

print(win_matrix)


for i in range(len(heim)):
    heim_player = heim[heim_indices[i]]
    gast_player = gast[gast_indices[i]]
    print(f"{heim_player} vs {gast_player}: {win_matrix.loc[gast_player, heim_player]:.2f}")

# print erwartete punkte
erw = sum(win_matrix.loc[gast[gast_indices[i]], heim[heim_indices[i]]] for i in range(len(heim)))
print(f"Erwartete Punkte: {erw:.2f}")

                          HAUSEGGER Andrea  GSTREIN Franziska  \
OBERHOFER Nina                    0.820242           0.688865   
NEMETH Enikö                      0.787044           0.648349   
NIKOLIC Maja Katharina            0.689141           0.514284   
ZATSCHKOWITSCH Katharina          0.604509           0.438664   
PRÖLL Tanja                       0.658793           0.493969   
STÖCKL Sandra                     0.582468           0.429258   

                          POCKSTALLER Katharina  RABANSER Vera  \
OBERHOFER Nina                         0.791190       0.769390   
NEMETH Enikö                           0.768703       0.721991   
NIKOLIC Maja Katharina                 0.663976       0.603617   
ZATSCHKOWITSCH Katharina               0.597877       0.524964   
PRÖLL Tanja                            0.648662       0.581426   
STÖCKL Sandra                          0.557631       0.505244   

                          KOLLENDA Laura  KUNOVA Nikola  
OBERHOFER Nina         

In [ ]:
# # players = ["ANGENBAUER Angelika", "NIKOLIC Maja Katharina", "PRÖLL Tanja", "STÖCKL Sandra", "ERTL Maria", "HORVATH Renate"]
# players = ["ANGENBAUER Angelika", "NIKOLIC Maja Katharina", "PRÖLL Tanja", "STÖCKL Sandra", "PYCHOVA Barbora", "HORVATH Renate"]

# # opponents = ["JEZSOVICZKI Anna", "RATASICH Theresia", "JANITS Klara", "PANTZER Eva", "NEMETH Enikö", "ARTNER Elfriede"]
# opponents = ["BLÜMEL Angelika", "STREBINGER Angela", "KUTI Alexandra", "BROEKMANN Sabine", "HUDI Zsofia", "WEISS Romana"]
# opponents = ["HUBER Julia", "LENGAUER Monika", "PETRU Thea", "MEISTERL Jasmin", "BABICKA Renata", "HAUER Dominique"]
# opponents = ["KOLLENDA Laura", "GRUBER Melissa", "WINDHOFER Sophie", "RABANSER Vera", "GSTREIN Franziska", "HAUSEGGER Andrea"]
# win_matrix = win_prob_matrix(players, opponents)
# win_matrix

In [ ]:
row_ind, col_ind = linear_sum_assignment(-win_matrix)

for i, j in zip(row_ind, col_ind):
    print(f"{players[i]} vs {opponents[j]}: {win_matrix.iloc[i, j]:.2f}")

ANGENBAUER Angelika vs WINDHOFER Sophie: 0.74
NIKOLIC Maja Katharina vs HAUSEGGER Andrea: 0.69
PRÖLL Tanja vs KOLLENDA Laura: 0.62
STÖCKL Sandra vs GSTREIN Franziska: 0.42
PYCHOVA Barbora vs GRUBER Melissa: 0.80
HORVATH Renate vs RABANSER Vera: 0.47
